# The handoff: a pooled posterior becomes a `core.Prior`

Evidence pooled across studies re-enters the next study as its prior. `prior_from_pool` turns a
`PoolResult` into a `core.Prior` plus the `LedgerLine` that records where it came from (the
pool's model hash, the target, the numbers). Two targets, selected by `PriorTarget`:

| target | prior | when |
|---|---|---|
| `"mu"` | `N(E[mu], sd[mu])` | the new study asks "what is the typical effect" |
| `"predictive"` | `N(E[mu], sqrt(E[tau²] + sd[mu]²))`, `E[tau²] = E[tau]² + sd[tau]²` | the new study is another draw from the family (the usual, wider case) |

`meta` returns core objects only; it never imports `calibrate` or `surface`. This notebook shows
the prior plugged into a `surface` kernel's `amplitude_prior`.

In [ ]:
import numpy as np

from axiom.core import D, LedgerLine, Outcome, Prior, Treatment
from axiom.meta import (
    Corpus, Pooled, PoolPriors, PoolSpec, PriorTarget, StudyRecord, pool, prior_from_pool,
)
from axiom.surface import GeometricCarryover, HillKernel, SurfaceSpec, build

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

## A pool on the natural scale

Twelve experimental reads of an elasticity; the pool is fitted with a free `tau`.

In [ ]:
rng = np.random.default_rng(3)
G = 12
se = rng.uniform(0.05, 0.2, G)
y = rng.normal(0.6, 0.15, G) + se * rng.normal(size=G)
corpus = Corpus(records=tuple(
    StudyRecord(study=f"s{g}", contributor=f"c{g}", quantity="elasticity", estimate=float(y[g]),
                se=float(se[g]), read="experiment", family="fertilizer")
    for g in range(G)
))
out = pool(PoolSpec(family="fertilizer", priors=PoolPriors(mu_scale=2.0, tau_scale=0.5)), corpus, draws=1000, seed=0)
assert isinstance(out, Pooled)
res = out.result
print(f"mu {res.mu.mean:.3f} ± {res.mu.sd:.3f} | tau {res.tau.mean:.3f} ± {res.tau.sd:.3f}")

In [ ]:
rows = []
for target in ("mu", "predictive"):
    t: PriorTarget = target
    prior, line = prior_from_pool(res, target=t, family="normal")
    assert isinstance(prior, Prior) and isinstance(line, LedgerLine)
    rows.append([target, f"{prior.family} {prior.hyper}", line.kind, line.statement])
table(rows, headers=("target", "prior", "ledger kind", "statement"))
print("ledger detail keys:", sorted(line.detail)[:6], "... | source = pool model hash:", line.source[:12])

## Into a surface kernel

A kernel's `amplitude_prior` must have positive support (`halfnormal`, `lognormal`, `gamma`), so
the handoff asks for `family="lognormal"`. On a natural-scale pool the lognormal is
moment-matched (`sigma² = log(1 + s²/m²)`, `mu = log m − sigma²/2`), preserving the pooled mean
and sd; on a log-scale pool (a `response_ratio`) the normal on the log scale *is* the lognormal.
`build` shows the prior sitting on the amplitude parameter `beta_a` of the assembled model.

In [ ]:
amp_prior, amp_line = prior_from_pool(res, target="predictive", family="lognormal")
print(amp_prior, "|", amp_line.detail["lognormal"])
m, s = np.exp(amp_prior.hyper["mu"] + amp_prior.hyper["sigma"] ** 2 / 2), None
print("lognormal mean matches the pooled predictive mean:", round(float(m), 4), "vs", round(res.mu.mean, 4))

spec = SurfaceSpec(
    name="next-study",
    treatments=(Treatment(name="a", dimension=D.currency, unit="USD"),),
    outcome=Outcome(name="y", dimension=D.outcome),
    kernels={"a": HillKernel(reference_dose=50.0, amplitude_prior=amp_prior)},
    carryover={"a": GeometricCarryover(max_lag=4)},
)
model = build(spec)
beta = model.parameter("beta_a")
print("amplitude parameter prior in the built model:", beta.prior)

In [ ]:
ratios = Corpus(records=tuple(
    StudyRecord(study=f"r{g}", contributor=f"c{g}", quantity="response_ratio",
                estimate=float(np.exp(rng.normal(0.3, 0.1))), se=0.08, read="experiment", family="fertilizer")
    for g in range(8)
))
log_pool = pool(PoolSpec(family="fertilizer", priors=PoolPriors(mu_scale=2.0)), ratios, draws=1000, seed=1)
assert isinstance(log_pool, Pooled)
print("pool scale:", log_pool.result.scale, "| mu (a log ratio):", round(log_pool.result.mu.mean, 3))
p, ln = prior_from_pool(log_pool.result, target="mu", family="lognormal")
print(p, "|", ln.detail["lognormal"])